# 11. 串流輸出

學習如何在 LangGraph 中實現即時串流輸出。

---

## 🎯 學習目標

完成本章節後，您將能夠：
- ✅ 使用 `stream()` 觀察執行過程
- ✅ 理解不同的 stream_mode
- ✅ 實現多節點串流
- ✅ 處理串流資料

---

## 📊 串流模式比較

```
┌─────────────────────────────────────────────────────────┐
│                    串流模式比較                          │
├─────────────────────────────────────────────────────────┤
│                                                         │
│   invoke()                                              │
│   ┌────────────────────────────┐                       │
│   │ 等待... 等待... 等待... ✅  │ ──▶ 最終結果         │
│   └────────────────────────────┘                       │
│                                                         │
│   stream()                                              │
│   ┌────┐ ┌────┐ ┌────┐ ┌────┐                          │
│   │ A  │ │ B  │ │ C  │ │ ✅ │ ──▶ 即時更新            │
│   └────┘ └────┘ └────┘ └────┘                          │
│     ▼      ▼      ▼      ▼                             │
│    UI     UI     UI     UI                             │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

### stream_mode 選項

| Mode | 返回內容 | 適用場景 |
|------|---------|----------|
| `updates` | 每個節點的更新 | 追蹤進度 |
| `values` | 完整狀態快照 | 完整狀態 |

In [1]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

---

## 11.1 基本串流

In [2]:
class ChatState(TypedDict):
    """對話狀態"""
    messages: Annotated[list, add_messages]

def chatbot(state: ChatState) -> dict:
    """模擬聊天機器人"""
    return {"messages": [{"role": "assistant", "content": "這是模擬回覆"}]}

graph = StateGraph(ChatState)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

app = graph.compile()
print("✅ 串流聊天機器人已就緒")

✅ 串流聊天機器人已就緒


In [3]:
print("📊 基本串流（updates 模式）：")
print("=" * 50)

for chunk in app.stream({"messages": [{"role": "user", "content": "你好"}]}):
    for node_name, values in chunk.items():
        print(f"\n📍 節點: {node_name}")
        print(f"   更新的欄位: {list(values.keys())}")
        if "messages" in values:
            print(f"   最新訊息: {values['messages'][-1]}")

📊 基本串流（updates 模式）：

📍 節點: chatbot
   更新的欄位: ['messages']
   最新訊息: {'role': 'assistant', 'content': '這是模擬回覆'}


---

## 11.2 多節點串流

In [4]:
class PipelineState(TypedDict):
    """管道狀態"""
    input: str
    step1: str
    step2: str
    step3: str
    output: str

def step_1(state: PipelineState) -> dict:
    print("  🔄 步驟 1 執行中...")
    return {"step1": f"[步驟1處理] {state['input']}"}

def step_2(state: PipelineState) -> dict:
    print("  🔄 步驟 2 執行中...")
    return {"step2": f"[步驟2處理] {state['step1']}"}

def step_3(state: PipelineState) -> dict:
    print("  🔄 步驟 3 執行中...")
    return {"step3": f"[步驟3處理] {state['step2']}", "output": "完成"}

# 建構多步驟管道
pipeline = StateGraph(PipelineState)
pipeline.add_node("step1", step_1)
pipeline.add_node("step2", step_2)
pipeline.add_node("step3", step_3)

pipeline.add_edge(START, "step1")
pipeline.add_edge("step1", "step2")
pipeline.add_edge("step2", "step3")
pipeline.add_edge("step3", END)

pipeline_app = pipeline.compile()
print("✅ 多節點管道已就緒")

✅ 多節點管道已就緒


In [5]:
print("📊 多節點串流：")
print("=" * 50)

initial_state = {"input": "測試資料", "step1": "", "step2": "", "step3": "", "output": ""}

for i, chunk in enumerate(pipeline_app.stream(initial_state), 1):
    for node_name, values in chunk.items():
        print(f"\n[{i}] ✓ 節點 '{node_name}' 完成")
        print(f"    更新: {values}")

📊 多節點串流：
  🔄 步驟 1 執行中...

[1] ✓ 節點 'step1' 完成
    更新: {'step1': '[步驟1處理] 測試資料'}
  🔄 步驟 2 執行中...

[2] ✓ 節點 'step2' 完成
    更新: {'step2': '[步驟2處理] [步驟1處理] 測試資料'}
  🔄 步驟 3 執行中...

[3] ✓ 節點 'step3' 完成
    更新: {'step3': '[步驟3處理] [步驟2處理] [步驟1處理] 測試資料', 'output': '完成'}


---

## 11.3 stream_mode 比較

In [6]:
print("📊 stream_mode='values' 模式：")
print("=" * 50)
print("返回每個步驟後的完整狀態快照\n")

for i, state in enumerate(pipeline_app.stream(
    {"input": "test", "step1": "", "step2": "", "step3": "", "output": ""},
    stream_mode="values"
)):
    print(f"[{i}] 完整狀態: {state}")

📊 stream_mode='values' 模式：
返回每個步驟後的完整狀態快照

[0] 完整狀態: {'input': 'test', 'step1': '', 'step2': '', 'step3': '', 'output': ''}
  🔄 步驟 1 執行中...
[1] 完整狀態: {'input': 'test', 'step1': '[步驟1處理] test', 'step2': '', 'step3': '', 'output': ''}
  🔄 步驟 2 執行中...
[2] 完整狀態: {'input': 'test', 'step1': '[步驟1處理] test', 'step2': '[步驟2處理] [步驟1處理] test', 'step3': '', 'output': ''}
  🔄 步驟 3 執行中...
[3] 完整狀態: {'input': 'test', 'step1': '[步驟1處理] test', 'step2': '[步驟2處理] [步驟1處理] test', 'step3': '[步驟3處理] [步驟2處理] [步驟1處理] test', 'output': '完成'}


In [7]:
print("📊 stream_mode='updates' 模式（預設）：")
print("=" * 50)
print("只返回每個節點的更新內容\n")

for i, update in enumerate(pipeline_app.stream(
    {"input": "test", "step1": "", "step2": "", "step3": "", "output": ""},
    stream_mode="updates"
)):
    print(f"[{i}] 更新: {update}")

📊 stream_mode='updates' 模式（預設）：
只返回每個節點的更新內容

  🔄 步驟 1 執行中...
[0] 更新: {'step1': {'step1': '[步驟1處理] test'}}
  🔄 步驟 2 執行中...
[1] 更新: {'step2': {'step2': '[步驟2處理] [步驟1處理] test'}}
  🔄 步驟 3 執行中...
[2] 更新: {'step3': {'step3': '[步驟3處理] [步驟2處理] [步驟1處理] test', 'output': '完成'}}


---

## 11.4 進度追蹤範例

In [8]:
import time

class ProgressState(TypedDict):
    """進度追蹤狀態"""
    task: str
    progress: int
    status: str

def phase_1(state):
    time.sleep(0.3)  # 模擬耗時
    return {"progress": 33, "status": "資料收集中..."}

def phase_2(state):
    time.sleep(0.3)
    return {"progress": 66, "status": "處理中..."}

def phase_3(state):
    time.sleep(0.3)
    return {"progress": 100, "status": "完成！"}

progress_graph = StateGraph(ProgressState)
progress_graph.add_node("phase1", phase_1)
progress_graph.add_node("phase2", phase_2)
progress_graph.add_node("phase3", phase_3)

progress_graph.add_edge(START, "phase1")
progress_graph.add_edge("phase1", "phase2")
progress_graph.add_edge("phase2", "phase3")
progress_graph.add_edge("phase3", END)

progress_app = progress_graph.compile()
print("✅ 進度追蹤器已就緒")

✅ 進度追蹤器已就緒


In [9]:
print("📊 即時進度追蹤：")
print("=" * 50)

for chunk in progress_app.stream({"task": "處理任務", "progress": 0, "status": "開始"}):
    for node, values in chunk.items():
        progress = values.get("progress", 0)
        status = values.get("status", "")
        bar = "█" * (progress // 10) + "░" * (10 - progress // 10)
        print(f"  [{bar}] {progress}% - {status}")

📊 即時進度追蹤：


  [███░░░░░░░] 33% - 資料收集中...


  [██████░░░░] 66% - 處理中...


  [██████████] 100% - 完成！


---

## 💡 重點回顧

### stream() vs invoke()

```python
# invoke: 等待全部完成
result = app.invoke(input)

# stream: 即時獲取更新
for chunk in app.stream(input):
    process(chunk)
```

### chunk 結構

```python
# updates 模式
chunk = {"node_name": {"field": "updated_value"}}

# values 模式
chunk = {"field1": "value1", "field2": "value2", ...}
```

### 應用場景

| 場景 | 用途 |
|------|------|
| 進度條 | 顯示處理進度 |
| 即時日誌 | 展示執行過程 |
| 打字機效果 | 逐字顯示回覆 |
| 監控面板 | 追蹤系統狀態 |

---

## 📝 練習題

1. **進度百分比**：建立更細緻的進度顯示（每 10%）
2. **執行時間**：在串流中追蹤每個節點的執行時間
3. **取消機制**：實作可以中途取消的串流
4. **多工串流**：同時處理多個任務並串流結果

---

下一步：[12. 子圖設計](12_subgraphs.ipynb)